# 第五课｜布尔逻辑、条件选择与时钟寄存器

数字电路需要计算条件、选择数据，并在指定时刻保存结果。本课从 AND、OR、NOT 的真值表开始，读一个完整的 SystemVerilog 累加器，逐项解释 `always_ff @(posedge clk)`。


## 1. 本课使用的概念

**布尔逻辑（Boolean logic）**处理真假条件，用 0 表示假、1 表示真。**组合逻辑（combinational logic）**根据当前输入计算结果；**寄存器（register）**保存多位状态；**时钟上升沿（positive clock edge）**是已知 0/1 时钟从 0 变为 1 的事件。第四课讲过状态更新，这一课将条件计算和保存时刻写到具体语句上。


## 2. NOT、AND、OR 与真值表

NOT（非）反转一个条件；AND（与）要求两个条件都真；OR（或）要求至少一个条件为真，包括两个都真的情况。

| a | b | NOT a | a AND b | a OR b |
|---|---|---|---|---|
| 0 | 0 | 1 | 0 | 0 |
| 0 | 1 | 1 | 0 | 1 |
| 1 | 0 | 0 | 0 | 1 |
| 1 | 1 | 0 | 1 | 1 |

这叫**真值表（truth table）**，它列出全部输入组合。下面 Python 的 `False/True` 对应 0/1，`not/and/or` 对应非/与/或。`product(..., repeat=2)` 枚举两个输入的四种组合。先预测 `a=False,b=True` 这一行，再运行。


In [ ]:
from itertools import product

def NOT(a): return not a
def AND(a, b): return a and b
def OR(a, b): return a or b

print(' a     b   NOT a   a AND b   a OR b')
for a, b in product([False, True], repeat=2):
    print(f'{a!s:5} {b!s:5} {NOT(a)!s:7} {AND(a,b)!s:9} {OR(a,b)!s:8}')


## 3. 比较器输出也可以参与逻辑运算

**比较器（comparator）**比较两个数，输出一个真假条件。`candidate_v >= threshold` 包含等于阈值的情况。若还要求使能信号 `enable` 为真，则发放条件为：

`spike_condition = enable AND (candidate_v >= threshold)`

| enable | candidate_v | threshold | reached | spike_condition |
|---|---:|---:|---|---|
| 0 | 4 | 4 | 1 | 0 |
| 1 | 3 | 4 | 0 | 0 |
| 1 | 4 | 4 | 1 | 1 |

先预测最后一行改用 `>` 后会怎样。


In [ ]:
threshold = 4
for enable, value in [(False, 4), (True, 3), (True, 4)]:
    reached = value >= threshold
    spike_condition = enable and reached
    print(f'enable={enable}, value={value}, reached={reached}, spike_condition={spike_condition}')


## 4. 条件怎样选择待保存的数据

**多路选择器（multiplexer，常简称 mux）**从多个输入值中选一个作为输出。若发放后保存 0，否则保存候选电位，关系是：

`value_to_store = 0 if spike_condition else candidate_v`

比较器产生 `reached`；AND 结合 `enable` 得到发放条件；选择器据此选出数据；寄存器在时钟沿保存所选值。比较、逻辑运算和数据选择属于组合逻辑。


## 5. 把状态更新写成 SystemVerilog

**硬件描述语言（Hardware Description Language, HDL）**用文字描述数字硬件的结构和行为。**SystemVerilog**是一种 HDL，同时支持仿真验证。下面使用仓库中的累加器：每个上升沿，复位有效时清零，否则把输入加到旧状态。

```systemverilog
module clocked_accumulator (
    input  logic              clk,
    input  logic              rst_n,
    input  logic signed [7:0] input_value,
    output logic signed [7:0] state
);
    always_ff @(posedge clk) begin
        if (!rst_n)
            state <= '0;
        else
            state <= state + input_value;
    end
endmodule
```

`module` / `endmodule` 声明模块的开始和结束；`clocked_accumulator` 是用户起的模块名。括号内声明**端口（port）**：`input` 是输入，`output` 是输出。三个输入是 `clk`、`rst_n`、`input_value`，输出是 `state`。

`logic` 是四态数据类型；`signed` 指定有符号解释；`[7:0]` 指定编号 7 到 0 的八位。这里 `state` 的存储结构由后面的时钟赋值决定，不能只看 `logic` 就判断它是寄存器。8 位补码范围为 −128 到 127，本例超出范围时保留低 8 位。


### `always_ff @(posedge clk)` 逐项解释

**`always_ff` 是 SystemVerilog 的标准关键字，也是保留字。** 它由语言定义，不能当作普通变量名；这里没有调用自定义函数。`ff` 来自 flip-flop（触发器）：一个触发器保存一位，8 个触发器可组成一个 8 位寄存器。

| 写法 | 语法含义 | 本例中的作用 |
|---|---|---|
| `always_ff` | 描述时序行为的过程块 | 描述 `state` 寄存器的更新 |
| `@(...)` | 事件控制：等待指定事件后执行块内语句 | 指定更新时刻 |
| `posedge` | 标准关键字，表示上升沿 | 在已知的 0/1 时钟中是 0→1 |
| `clk` | 用户定义的信号名 | 本模块的时钟输入 |
| `begin` / `end` | 把多条语句组成一个块 | 组织复位和累加分支 |

在仿真中，过程等待 `clk` 的上升沿，执行块内语句，再等待下一次上升沿。在本例对应的电路中，加法器随输入和旧状态变化而计算；寄存器只在上升沿采样要保存的值。时钟保持高电平、保持低电平或发生下降沿时，`state` 保持原值。实际电路的输出在边沿之后经过传播延迟才稳定。

SystemVerilog 的 `logic` 还可表示 `X`（未知）和 `Z`（高阻）。四态仿真的 `posedge` 也包含 0→X/Z、X/Z→1；本课时序表使用已知的 0/1 时钟。


### `if (!rst_n)`、`'0` 和 `<=`

`if` / `else` 是条件语句关键字。`!` 是逻辑非；当 `rst_n=0` 时，`!rst_n=1`，选择复位分支。`rst_n` 是用户起的名字，`_n` 是常见的低有效命名约定，名字本身不会改变电路行为。

`'0` 是填满赋值目标位宽的全零常量；这里给 8 位 `state` 写入 `00000000`。`<=` 在赋值语句中表示**非阻塞赋值（nonblocking assignment）**，在比较表达式中则表示“小于等于”，要看所在语法位置。

在上升沿，`state <= state + input_value;` 先用旧 `state` 和当时的输入计算右侧结果，再安排在当前仿真时间步的非阻塞赋值更新阶段写入左侧。例如旧值为 3、输入为 2，边沿后保存 5。它不是等到下一个周期才赋值。

若同一时钟块写 `a <= b; b <= a;`，且两个变量只有该块写入，那么两条右侧都读取更新前的值，边沿后交换原来的值。组合计算通常使用阻塞赋值 `=`，使后面的语句能使用前面刚算出的中间结果。

这里是**同步、低有效复位**：`rst_n=0` 只有在上升沿才让 `state` 清零。若在两个上升沿之间把它拉低，状态先保持，到下一个上升沿才清零。异步复位的常见写法是 `@(posedge clk or negedge rst_n)`，复位拉低也触发该块；这是另一种复位时序。


## 6. 对应的电路和边沿前后数值

`state` 的输出和 `input_value` 送入加法器，得到和。`!rst_n` 控制选择器：复位时选择 0，否则选择和。选择器输出接到 8 位寄存器的数据输入，`clk` 接到时钟输入。寄存器输出同时送回加法器。

<svg xmlns="http://www.w3.org/2000/svg" viewBox="0 0 840 300" role="img" aria-label="l05-state-update" style="width:100%;height:auto">
<defs><marker id="l05-state-update" markerWidth="8" markerHeight="8" refX="7" refY="4" orient="auto"><path d="M0,0 L8,4 L0,8 Z" fill="#506070"/></marker></defs>
<rect x="35" y="90" width="150" height="60" rx="6" fill="#e6f4ea" stroke="#506070"/><text x="110.0" y="125.0" text-anchor="middle" font-family="sans-serif" font-size="15">input_value</text>
<rect x="240" y="90" width="130" height="60" rx="6" fill="#e6f4ea" stroke="#506070"/><text x="305.0" y="125.0" text-anchor="middle" font-family="sans-serif" font-size="15">adder</text>
<rect x="445" y="90" width="145" height="60" rx="6" fill="#e6f4ea" stroke="#506070"/><text x="517.5" y="125.0" text-anchor="middle" font-family="sans-serif" font-size="15">reset mux</text>
<rect x="660" y="90" width="155" height="60" rx="6" fill="#e6f4ea" stroke="#506070"/><text x="737.5" y="125.0" text-anchor="middle" font-family="sans-serif" font-size="15">state register</text>
<rect x="440" y="10" width="150" height="45" rx="6" fill="#e6f4ea" stroke="#506070"/><text x="515.0" y="37.5" text-anchor="middle" font-family="sans-serif" font-size="15">rst_n</text>
<rect x="660" y="210" width="155" height="45" rx="6" fill="#e6f4ea" stroke="#506070"/><text x="737.5" y="237.5" text-anchor="middle" font-family="sans-serif" font-size="15">clk</text>
<path d="M185 120 H240" fill="none" stroke="#506070" stroke-width="2" marker-end="url(#l05-state-update)"/>
<path d="M370 120 H445" fill="none" stroke="#506070" stroke-width="2" marker-end="url(#l05-state-update)"/>
<text x="407" y="110" text-anchor="middle" font-family="sans-serif" font-size="14">sum</text>
<path d="M590 120 H660" fill="none" stroke="#506070" stroke-width="2" marker-end="url(#l05-state-update)"/>
<text x="622" y="110" text-anchor="middle" font-family="sans-serif" font-size="14">data</text>
<path d="M515 55 V90" fill="none" stroke="#506070" stroke-width="2" marker-end="url(#l05-state-update)"/>
<path d="M737 210 V150" fill="none" stroke="#506070" stroke-width="2" marker-end="url(#l05-state-update)"/>
<text x="765" y="185" text-anchor="middle" font-family="sans-serif" font-size="14">clock</text>
<path d="M815 120 H830 V285 H305 V150" fill="none" stroke="#506070" stroke-width="2" marker-end="url(#l05-state-update)"/>
<text x="470" y="280" text-anchor="middle" font-family="sans-serif" font-size="14">stored state</text>
</svg>

已知旧状态为 3，输入为 2：

| 事件 | rst_n | state | 原因 |
|---|---|---:|---|
| 上升沿前 | 1 | 3 | 和为 5，尚未写入 |
| 上升沿后 | 1 | 5 | 采样累加结果 |
| 输入改成 7 | 1 | 5 | 加法结果变化，寄存器保持 |
| 下降沿 | 1 | 5 | 本例只在上升沿更新 |
| 两沿之间复位拉低 | 0 | 5 | 同步复位等待上升沿 |
| 下一个上升沿后 | 0 | 0 | 复位分支清零 |


## 7. Run：手算后检查边沿更新

下面的 Python 只计算每个上升沿后的值。先预测三次结果。`state + input_value` 均在 8 位范围内，Python 整数与本例 RTL 得到相同结果。


In [ ]:
state = 0
for edge, (rst_n, input_value) in enumerate([(False, 9), (True, 2), (True, 3)]):
    before = state
    state = 0 if not rst_n else state + input_value
    print(f'edge={edge}, before={before}, rst_n={rst_n}, input={input_value}, after={state}')


### 三种过程块与工具检查

Verilog-2001 已有通用 `always`，例如 `always @(posedge clk)`；SystemVerilog 保留它，并增加以下标准关键字：

| 关键字 | 用途 | 读代码时检查什么 |
|---|---|---|
| `always_ff` | 触发器等时序行为 | 更新事件、复位、写入的寄存器 |
| `always_comb` | 组合行为 | 所有输出在每条分支中都有赋值 |
| `always_latch` | 电平敏感锁存行为 | 使能有效时传递数据，其他时候保持 |

`always_ff` 要求恰好一个事件控制，并禁止阻塞式时间控制；它写入的变量不能由其他过程再写入。一个 `@(posedge clk or negedge rst_n)` 仍是一个事件控制，其中包含两个事件。语言约束不能概括成“必须且只能写一个 `posedge`”。

`state + input_value` 在 `always_ff` 右侧仍对应组合加法器；这个块描述何时保存结果，并不表示块内只能有存储单元。工具可以检查时序建模意图，具体报警取决于工具，不能保证所有错误都自动报错。

`always_comb` 自动确定相关读取信号的敏感性，并在仿真开始时执行一次。**自动敏感性不等于完整赋值**：若写 `if (enable) y = d;` 而未给 `enable=0` 的情况赋值，`y` 需要保持旧值，会形成锁存行为。写 `if (enable) y = d; else y = 0;` 才给出了两种情况的组合结果。


## 8. Try It

1. 旧状态为 6、输入为 −2、复位无效，下一上升沿后是什么？
2. 保持 `clk=1`，只改变输入，为什么寄存器不更新？
3. 在下降沿前拉低同步复位，下次什么时候清零？
4. 对条件 `enable AND (candidate >= threshold)`，分别列出使能为 0、未达到、恰好达到阈值的例子。


## 9. 作业

[第05课作业：逻辑门、比较器与 enable](../../exercises/zh/05_logic_building_blocks.ipynb)

作业实现逻辑条件。本课新增的寄存器阅读练习用上一节的时序问题检查；第06课继续用实际编译工具读取这份累加器。


## 10. AI Task

让 AI 给累加器逐行加注释，区分关键字和用户名称，列出三个输入端口，并填写“边沿前状态、复位、输入、边沿后状态”的表。自己核对其同步复位和旧状态解释。


## 11. Human Check

独立说明 `always_ff`、`@`、`posedge`、`clk`、`<=`、`!rst_n`、`'0` 各自的含义。指出加法器、选择器和寄存器；解释自动敏感性为什么不能补齐遗漏的赋值。


## 12. Engineering Handoff

第06课将这里的硬件关系归纳为寄存器传输级（Register-Transfer Level, RTL），编译 `clocked_accumulator.sv`。第07课加入阈值和发放状态；第08课产生时钟、施加输入并自动检查输出。


## 13. 项目追踪 Project Trace

- Lesson: `LSN-005`
- Engineering slice: `RMD-003A`
- Prepares: `RMD-004`
- Learning requirement: `PFR1 / PDP1`
- 本例数值规则：8 位有符号回绕；正式神经元数值规范见 `RMD-002 / RMD-003`。

语法依据：[SystemVerilog 语言参考手册](https://courses.cs.washington.edu/courses/cse371/references/SystemVerilog_Reference_Manual.pdf)，第 9.2–9.4 节及关键字表。


## 14. Exit Ticket

用真值表解释 AND、OR、NOT；从累加器中标出组合计算和状态存储；逐字解释时钟事件控制；手算同步复位和累加两个分支。
